In [11]:
# Import libraries
import pandas as pd
import numpy as np

import joblib
import json # Used to pull our defined dictionaries 

import torch # Will use for tensor conversion and operation
from torch.utils.data import Dataset, DataLoader # Use for PyTorch Operations
import torch.nn as nn # This is what we use for our nn

from sklearn.preprocessing import StandardScaler # Use to transform continous features to zero mean
from sklearn.model_selection import train_test_split # Used for splitting our data
from sklearn.metrics import (average_precision_score, roc_auc_score, 
                             accuracy_score, matthews_corrcoef)

In [12]:
# Loading cleaned data and dictionaries
data = pd.read_csv('saved/nhanes_clean.csv')

with open('saved/col_lists.json', 'r') as f:
    cols = json.load(f)

demo_cols = cols['demo_cols']
demo_cols = [c for c in demo_cols if c not in ['SLQ300', 'SLQ310']] # dropping sleep times for now
disease_cols = cols['disease_cols']
demo_meta = cols['demo_meta']
disease_meta = cols['disease_meta']

In [13]:
# Standardising continuous demographic features
"""
Neural networks train much better when inputs are on similar scales
Binary and indicator cols (-1/0/1) are already on a reasonable scale
We scale continuous cols like age, BMI, SBP to zero mean unit variance
"""

continuous_cols = ['RIDAGEYR', 'BMXBMI', 'BMXWAIST', 'ALQ121', 
                   'PAQ706', 'OCQ180', 'HUQ010', 'SBP', 'DBP']

scaler = StandardScaler() # define scaler
data[continuous_cols] = scaler.fit_transform(data[continuous_cols]) # apply scaler using fit_transform

In [ ]:
# PyTorch Dataset
"""
PyTorch requires batches of tensors, so we use the following to convert from dataframe 
to a form PyTorch is happy with.

The way DataLoader works is:
For each epoch the class will
(1) Shuffle indices
(2) Grab 64 observations at a time
(3) Stack into tensors (X_batch = (64 × 22), D_batch = (64 × 15))
(4) Feed to model, compute loss, backprop
(5) The goes through the next batch
(6) Once goes through all batches ends
"""
class NHANESDataset(Dataset):
    def __init__(self, df, demo_cols, disease_cols):
        # Extract demo features as float tensor where shape is (n, d_demo)
        self.X = torch.tensor(df[demo_cols].astype(float).values, dtype=torch.float32)
        
        # Extract disease flags, where shape is (n, K)
        # NaN stays as NaN here, masked in loss function later
        self.D = torch.tensor(df[disease_cols].astype(float).values, dtype=torch.float32)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.D[idx]
        # returns (demo_features, disease_flags) for one participant
        # both demogprahic and disease are inputs and targets:
            # X goes into the trunk
            # D goes into heads as masked input and is the label to predict

In [15]:
# Splitting into training, validation, and test samples
"""
Let's begin with a 70/15/15 split:
  Train (70%): 5088  where the model learns weights from this
  Val   (15%): 1090  this is used for early stopping, hyperparameter tuning
  Test  (15%): 1091  final honest evaluation, only touched at the end
"""

train_df, temp_df = train_test_split(data, test_size=0.30, random_state=42) # trains keeps 70%
val_df,   test_df = train_test_split(temp_df, test_size=0.50, random_state=42) # splits remaining into two 15%s

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

# Wrapping each samples in Dataset and DataLoader
train_ds = NHANESDataset(train_df, demo_cols, disease_cols)
val_ds   = NHANESDataset(val_df,   demo_cols, disease_cols)
test_ds  = NHANESDataset(test_df,  demo_cols, disease_cols)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True) # shuffle for training
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False) # no shuffle for eval
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False)

# Sanity checks
X_batch, D_batch = next(iter(train_loader))
print(f"X batch shape: {X_batch.shape}")  # expect (64, 23)
print(f"D batch shape: {D_batch.shape}")  # expect (64, 15)
print(f"NaN in D batch: {torch.isnan(D_batch).sum().item()}") # expect some

Train: 5088, Val: 1090, Test: 1091
X batch shape: torch.Size([64, 23])
D batch shape: torch.Size([64, 15])
NaN in D batch: 177


In [16]:
# The neural network
"""
General structure will be as follows:
init: declares weights (random, untrained) - part of the class
forward: defines computation using those weights - part of the class
training: adjusts weights via backprop to minimise loss - setup as a standalone functio in next block
"""

class MaskedMTLNet(nn.Module):
    # The first function sets the stages declaring the architectural blue print
    def __init__(self, d_demo, K, H=128, dropout=0.3, circular_pairs=None):
        """
        d_demo: number of demographic input features (23)
        K: number of disease targets (15)
        H: hidden dimension of shared trunk
        circular_pairs: dict of {k: [list of head indices to also mask]}
        """
        super().__init__() # super refers to parent class (nn.Module here) which initialises its hidden set up
        """
        nn.Module does quite a few things, including:
        - sets up internal dictionaries to track parameters
        - sets up hooks for forward/backward passes
        - enables .to('cuda'), .train(), .eval() methods
        - enables torch.save() / torch.load()
        """
        self.K = K # store the arguments as instance variables
        self.circular_pairs = circular_pairs or {} # store the arguments as instance variables

        # The shared trunk - only takes demographic features for two hidden shared layers
        self.trunk = nn.Sequential(
            nn.Linear(d_demo, H),
            nn.BatchNorm1d(H),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(H, H),
            nn.BatchNorm1d(H),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # Task-specific heads
        # Each head k takes: concat(h_demo, d_minus_k)
        # d_minus_k = all disease flags except disease k (itself) and any circular pairs
        self.heads = nn.ModuleList()
        for k in range(K):
            n_masked = 1 + len(self.circular_pairs.get(k, []))  # self + circular pairs
            head_input_size = H + (K - n_masked) # adding 128 + non-(self + circular pairs)
            self.heads.append(nn.Sequential(
                nn.Linear(head_input_size, H // 2),
                nn.ReLU(),
                nn.Linear(H // 2, 1),
                nn.Sigmoid()
            ))

    def forward(self, x_demo, d):
        """
        x_demo: (batch, d_demo) - 2d tensor with no. of samples in batch and no.2 of demo features
        d: (batch, K) — 2d tensor with no. of samples and no.2 of disease features
        """
        # Shared trunk 
        h = self.trunk(x_demo) # this will be used for the demographic pass through layers

        # Following will focus on the pass through once diseases enter the neural network
        preds = [] # empty list which will collect each head's ouput
        for k in range(self.K):
            # Build the mask to show which disease columns to exclude for head k
            exclude = {k} | set(self.circular_pairs.get(k, [])) # always excludes itself + circular pairs
            keep = [j for j in range(self.K) if j not in exclude] # which to include (all minus exclude)
 
            d_input = d[:, keep].clone() # cloning to avoid amending original list
            d_input = torch.nan_to_num(d_input, nan=0.0) # replace NaN with 0 (unknown disease = assume absent)

            # Concatenate trunk output with masked disease context
            head_input = torch.cat([h, d_input], dim=1) # concatenate head along feature axis
            preds.append(self.heads[k](head_input)) # passes the conactenrate input theough the layers

        return torch.cat(preds, dim=1) # Stacks all 15 outputs into one tensor
    



In [ ]:
# The loss Function
"""
The loss function = Binary Cross Entropy (BCE) * Inverse Prevalance Weighting (IPW)

Two key features:
1. BCE - natural loss as derived from MLE for a bernoulli distribution
2. Class weights — IPW to handle class imbalance rare 
diseases (e.g. CHF 4.4%) get upweighted so model doesn't just predict 0 for everything
3. NaN masking — ignore participants with unknown disease status for head k
"""

# Compute class weights from training set
"""
IPW = 1 / prevalence(d_k)
Rare disease, means low prevalence, means high weight, and this loss penalises mistakes more
The prevalence had a very big range (10x) so have rescaled to [1,3] as otherwise too much 
focus on the rare diseases.
"""

disease_weights = []
for col in disease_cols:
    prev = train_df[col].mean() # prevalence = proportion positive
    disease_weights.append(1.0 / prev)

# Rescale weights continuously - not used, but left in
raw = np.array(disease_weights)
w_min, w_max = raw.min(), raw.max()
disease_weights = list(1 + 0 * (raw - w_min) / (w_max - w_min)) # I opt for no rescaling, = 0
weights = torch.tensor(disease_weights, dtype=torch.float32)

# Check
for col, w in zip(disease_cols, disease_weights):
    print(f"  {disease_meta[col]:30s}  weight={w:.2f}")

def masked_weighted_bce_loss(predictions, targets, weights):
    """
    predictions: (batch, K) — models the output probabilities, all in (0,1)
    targets: (batch, K) — true disease labels, may contain NaN
    weights: (K,) — inverse prevalence weight per disease
    """
    total_loss = 0.0
    n_active = 0
    for k in range(predictions.shape[1]):
        p_k = predictions[:, k] # predicted probabilities for disease k (batch,)
        t_k = targets[:, k] # true labels for disease k (batch,)

        # Mask out NaN participants — don't penalise predictions where label unknown
        valid = ~torch.isnan(t_k) # boolean mask (batch,), True = has label
        if valid.sum() == 0: # if no valid labels for this disease, skip
            continue

        p_k = p_k[valid] # keep only valid predictions
        t_k = t_k[valid] # keep only valid labels

        # BCE loss for this head, weighted by inverse prevalence
        bce = nn.functional.binary_cross_entropy(p_k, t_k, reduction='mean') # calc loss as BCE
        total_loss += weights[k] * bce # weights here is the IPW required as input for function
        n_active += 1 # increment only if head is active (accounts for NaNs)

    return total_loss / n_active if n_active > 0 else torch.tensor(0.0) # average across active heads


  Diabetes                        weight=2.42
  Pre-diabetes                    weight=2.27
  Hypertension                    weight=1.07
  High cholesterol                weight=1.00
  Arthritis                       weight=1.09
  Congestive heart failure        weight=5.86
  Coronary heart disease          weight=4.41
  Heart attack                    weight=5.42
  Stroke                          weight=5.42
  Liver condition                 weight=4.16
  Thyroid problem                 weight=2.05
  COPD/emphysema                  weight=3.47
  Cancer                          weight=1.96
  Kidney disease                  weight=6.00
  Asthma                          weight=1.62


In [ ]:
# The training loop
"""
For each epoch we do the following:
1. Forward pass — model makes predictions
2. Loss — masked and weighted BCE is computed
3. Backward pass — gradients computed via backpropagation
4. Update — optimiser adjusts weights
5. Validate — we check performance on val set (only to see when to stop early in this version)
6. Early stop — stop if val PRAUC doesn't improve for patience = 20 epochs
"""

# Set hyperparameters (capitals)
EPOCHS = 100 # how many training loops
LR = 1e-3 # learning rate
PATIENCE = 20 # how many epochs to wait without improvement before stopping early
CIRCULAR_PAIRS = {0: [1], 1: [0]} # diabetes (idx 0) and pre-diabetes (idx 1) mask each other
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # try and use GPU

# Instantiate model and optimiser
model = MaskedMTLNet(d_demo=len(demo_cols), K=len(disease_cols), H=128, dropout=0.3,
                         circular_pairs=CIRCULAR_PAIRS).to(device)
# Opt for ADAM and a small weight_decay (L2 regularisation) to prevent overfitting:
optimiser = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4) 
weights = weights.to(device)

# Early stopping state
best_val_prauc = 0.0
patience_counter = 0
best_model_state = None

# Training
for epoch in range(EPOCHS):

    # Train phase
    model.train()  # activates dropout + batchnorm train mode
    train_loss = 0.0
    n_active = 0 

    for X_batch, D_batch in train_loader:
        X_batch = X_batch.to(device)
        D_batch = D_batch.to(device)

        optimiser.zero_grad() # clear gradients from previous batch
        preds = model(X_batch, D_batch) # forward pass
        loss  = masked_weighted_bce_loss(preds, D_batch, weights) # compute loss
        loss.backward() # backprop — compute gradients
        optimiser.step() # update weights
        train_loss += loss.item()

    train_loss /= len(train_loader) # average loss across batches

    # Validation phase
    model.eval() # deactivates dropout, batchnorm eval mode
    all_preds = []
    all_targets = []

    with torch.no_grad(): # no gradient tracking needed for eval
        for X_batch, D_batch in val_loader:
            X_batch = X_batch.to(device)
            D_batch = D_batch.to(device)
            preds   = model(X_batch, D_batch)
            all_preds.append(preds.cpu().numpy())
            all_targets.append(D_batch.cpu().numpy())

    all_preds = np.concatenate(all_preds,   axis=0)
    all_targets = np.concatenate(all_targets, axis=0) 

    # Macro average PRAUC across all disease heads
    prauc_scores = []
    for k in range(len(disease_cols)):
        valid = ~np.isnan(all_targets[:, k])
        if valid.sum() == 0 or all_targets[valid, k].sum() == 0:
            continue
        prauc_scores.append(
            average_precision_score(all_targets[valid, k], all_preds[valid, k])
        )
    val_prauc = np.mean(prauc_scores)

    print(f"Epoch {epoch+1:3d} | Train Loss: {train_loss:.4f} | Val PRAUC: {val_prauc:.4f}")

    # Early stopping
    if val_prauc > best_val_prauc:
        best_val_prauc   = val_prauc
        patience_counter = 0
        best_model_state = model.state_dict() # save best weights
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break

# Restore best model
model.load_state_dict(best_model_state)
print(f"\nBest Val PRAUC: {best_val_prauc:.4f}")

Epoch   1 | Train Loss: 0.9011 | Val PRAUC: 0.3138
Epoch   2 | Train Loss: 0.7215 | Val PRAUC: 0.3462
Epoch   3 | Train Loss: 0.7078 | Val PRAUC: 0.3574
Epoch   4 | Train Loss: 0.6913 | Val PRAUC: 0.3717
Epoch   5 | Train Loss: 0.6808 | Val PRAUC: 0.3807
Epoch   6 | Train Loss: 0.6693 | Val PRAUC: 0.3866
Epoch   7 | Train Loss: 0.6654 | Val PRAUC: 0.3875
Epoch   8 | Train Loss: 0.6605 | Val PRAUC: 0.3904
Epoch   9 | Train Loss: 0.6557 | Val PRAUC: 0.3917
Epoch  10 | Train Loss: 0.6526 | Val PRAUC: 0.3896
Epoch  11 | Train Loss: 0.6503 | Val PRAUC: 0.3911
Epoch  12 | Train Loss: 0.6457 | Val PRAUC: 0.3933
Epoch  13 | Train Loss: 0.6423 | Val PRAUC: 0.3945
Epoch  14 | Train Loss: 0.6442 | Val PRAUC: 0.3958
Epoch  15 | Train Loss: 0.6386 | Val PRAUC: 0.3942
Epoch  16 | Train Loss: 0.6416 | Val PRAUC: 0.3947
Epoch  17 | Train Loss: 0.6344 | Val PRAUC: 0.3974
Epoch  18 | Train Loss: 0.6328 | Val PRAUC: 0.3947
Epoch  19 | Train Loss: 0.6312 | Val PRAUC: 0.3951
Epoch  20 | Train Loss: 0.6243 

In [19]:
# Generate and save the test set predictions
model.eval()
test_preds, test_targets = [], []

with torch.no_grad():
    for X_batch, D_batch in test_loader:
        X_batch = X_batch.to(device)
        D_batch = D_batch.to(device)
        preds = model(X_batch, D_batch)
        test_preds.append(preds.cpu().numpy())
        test_targets.append(D_batch.cpu().numpy())

test_preds   = np.concatenate(test_preds,   axis=0)
test_targets = np.concatenate(test_targets, axis=0)

np.save('saved/mtl_test_preds.npy',   test_preds)
np.save('saved/mtl_test_targets.npy', test_targets)

# Saving the model
torch.save({
    'model_state_dict': best_model_state,
    'demo_cols': demo_cols,
    'disease_cols': disease_cols,
    'd_demo': len(demo_cols),
    'K': len(disease_cols),
    'H': 128,
    'dropout': 0.3,
    'circular_pairs': CIRCULAR_PAIRS,
}, 'saved/mtl_model.pt')

print("Predictions saved to mtl_test_preds.npy and model saved to mtl_model.pt")

Predictions saved to mtl_test_preds.npy and model saved to mtl_model.pt


In [ ]:
# saving the scaler in a pickle file to guarantee same scaler acorss notebook
joblib.dump(scaler, 'saved/demo_scaler.pkl')

['saved/demo_scaler.pkl']